# View the completed DotWall layout-transfer run

This notebook only reads saved results. It does **not** rerun the encoder or train either adapter. The spatial heatmaps were saved to Drive during the completed run, but the downloaded experiment notebook retained the metrics table without the inline figure outputs.


In [ ]:
from google.colab import drive
from pathlib import Path
from IPython.display import display, Image, Markdown
import json

drive.mount('/content/drive')
RUN_ID = '20261110T125759955799Z'
ROOT = Path('/content/drive/MyDrive/datasets/curriculum_cost_adapter/dotwall_layout_transfer_v1') / RUN_ID
PLOTS = ROOT / 'both_unseen' / 'heatmaps'
assert PLOTS.is_dir(), f'Missing heatmap folder: {PLOTS}'
images = sorted(PLOTS.glob('wall*_door*_goal*.png'))
print('Run folder:', ROOT)
print('Matched raw / fixed / multi heatmaps:', len(images))
assert len(images) == 16, 'Expected four goals on each of four unseen layouts.'


## Compare the same goal across all four unseen layouts

The five panels in each image are geodesic distance, straight-line distance, raw I-JEPA MSE, fixed-adapter MSE, and multi-layout-adapter MSE. Each panel has its own color scale. Set `GOAL_NUMBER` to 1–4 and rerun this cell to inspect another goal.

1. `(14, 22)` — left, upper
2. `(14, 40)` — left, lower
3. `(50, 22)` — right, upper
4. `(50, 40)` — right, lower


In [ ]:
GOAL_NUMBER = 3
assert GOAL_NUMBER in (1, 2, 3, 4)
for wall_x, door_y in ((26, 22), (26, 38), (38, 22), (38, 38)):
    path = PLOTS / f'wall{wall_x:02d}_door{door_y:02d}_goal{GOAL_NUMBER}.png'
    assert path.is_file(), path
    display(Markdown(f'### Wall {wall_x}, doorway {door_y}'))
    display(Image(filename=str(path), width=1700))


## Changed-order diagnostic

The main transfer question is whether each cost changes its ordering when moving the wall or door reverses the geodesic ordering of the same query pair. This table summarizes the saved standard-to-unseen comparison. The query pairs within a layout are correlated, so use this as a diagnostic rather than an independent-sample significance test.


In [ ]:
path = ROOT / 'both_unseen' / 'changed_order_vs_standard.json'
assert path.is_file(), path
comparisons = json.loads(path.read_text())
flips = sum(row['truth_flips'] for row in comparisons)
print('Matched standard-to-unseen cases:', len(comparisons))
print('True geodesic order reversals:', flips)
for model in ('raw', 'fixed', 'multi'):
    changed = sum(row['models'][model]['changed'] for row in comparisons)
    correct = sum(row['models'][model]['correct_both'] for row in comparisons)
    ties = sum(row['models'][model]['ties'] for row in comparisons)
    if flips:
        print(f'{model:5s} changed={changed:,} ({changed/flips:.1%}), correct in both layouts={correct:,} ({correct/flips:.1%}), ties={ties:,}')
    else:
        print(f'{model:5s}: no true order reversals in these cases')
